In [1]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
df = pd.read_csv("./korean_skincare_master.csv")

In [4]:
# Helper function to convert product & brand names into a clean URL slug
def slugify(text):
  text = str(text).lower().strip()
  # Remove special characters like +, :, etc.
  text = re.sub(r"[^\w\s-]", "", text)
  # Replace spaces and multiple dashes with a single dash
  text = re.sub(r"[\s_-]+", "-", text)
  return text


def scrape_skinsort_rating(brand, product_name, session):
  brand_slug = slugify(brand)
  product_slug = slugify(product_name)

  # Direct product URL pattern on SkinSort
  url = f"https://skinsort.com/products/{brand_slug}/{product_slug}"

  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
          " like Gecko) Chrome/124.0.0.0 Safari/537.36"
      ),
      "Accept-Language": "en-US,en;q=0.9",
  }

  try:
    response = session.get(url, headers=headers, timeout=10)
    if response.status_code == 200:
      soup = BeautifulSoup(response.text, "html.parser")

      # Locate the review link anchor shown in your inspector screenshot
      review_anchor = soup.find(
          "a", href=lambda h: h and "/reviews" in h and "/products/" in h
      )
      if review_anchor:
        rating_span = review_anchor.find(
            "span", class_=lambda c: c and "font-semibold" in c
        )
        if rating_span and rating_span.text.strip():
          rating_val = float(rating_span.text.strip())
          return rating_val

      # Fallback: find any span directly containing the rating format (e.g. 4.4)
      rating_elem = soup.find(
          "span", class_=re.compile(r"font-semibold.*tracking-tight")
      )
      if rating_elem:
        val = rating_elem.text.strip()
        match = re.search(r"\d+\.\d+", val)
        if match:
          rating_val = float(match.group(0))
          return rating_val

    return None

  except Exception as e:
    return None

# 1. Prepare the top 100 products
top_100_products = (
    df.sort_values(by="Review_Count", ascending=False)
    .head(100)[["Product Name", "Brand", "Review_Count", "Rating"]]
    .reset_index(drop=True)
)

top_100_products["Skinsort_Rating"] = None

# 2. Scrape with a persistent session and polite rate limiting
session = requests.Session()

for idx, row in top_100_products.iterrows():
  brand = row["Brand"]
  product = row["Product Name"]

  rating = scrape_skinsort_rating(brand, product, session)
  top_100_products.at[idx, "Skinsort_Rating"] = rating

  # Pause 1-2 seconds between requests to avoid IP bans / 429 Too Many Requests
  time.sleep(1.5)

print("Scraping completed!")
print(top_100_products.head(10))

Scraping completed!
                                    Product Name             Brand  \
0  Relief Sun: Rice + Niacinamide SPF 50+ PA++++  BEAUTY OF JOSEON   
1          Advanced Snail 96 Mucin Power Essence             COSRX   
2                Glow Serum Propolis+Niacinamide  BEAUTY OF JOSEON   
3                                  Dynasty Cream  BEAUTY OF JOSEON   
4                               345 Relief Cream         DR.ALTHEA   
5                      Master Patch Original Fit             COSRX   
6           Heartleaf Pore Control Cleansing Oil              ANUA   
7           Revive Eye Serum : Ginseng + Retinal  BEAUTY OF JOSEON   
8                   Heartleaf 77% Soothing Toner              ANUA   
9           Aloe Soothing Sun Cream SPF50+ PA+++             COSRX   

   Review_Count  Rating Skinsort_Rating  
0        7298.0    4.75            None  
1        2736.0    4.65             3.4  
2        2663.0    4.60            None  
3        2645.0    4.62             2.8  

In [5]:
# Helper function to convert product & brand names into a clean URL slug
def slugify(text):
  text = str(text).lower().strip()
  # Remove special characters like +, :, etc.
  text = re.sub(r"[^\w\s-]", "", text)
  # Replace spaces and multiple dashes with a single dash
  text = re.sub(r"[\s_-]+", "-", text)
  return text


def scrape_skinsort_rating(brand, product_name, session):
  brand_slug = slugify(brand)
  product_slug = slugify(product_name)

  # Direct product URL pattern on SkinSort
  url = f"https://skinsort.com/products/{brand_slug}/{product_slug}"

  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
          " like Gecko) Chrome/124.0.0.0 Safari/537.36"
      ),
      "Accept-Language": "en-US,en;q=0.9",
  }

  try:
    response = session.get(url, headers=headers, timeout=10)
    if response.status_code == 200:
      soup = BeautifulSoup(response.text, "html.parser")

      # Locate the review link anchor shown in your inspector screenshot
      review_anchor = soup.find(
          "a", href=lambda h: h and "/reviews" in h and "/products/" in h
      )
      if review_anchor:
        rating_span = review_anchor.find(
            "span", class_=lambda c: c and "font-semibold" in c
        )
        if rating_span and rating_span.text.strip():
          rating_val = float(rating_span.text.strip())
          return rating_val

      # Fallback: find any span directly containing the rating format (e.g. 4.4)
      rating_elem = soup.find(
          "span", class_=re.compile(r"font-semibold.*tracking-tight")
      )
      if rating_elem:
        val = rating_elem.text.strip()
        match = re.search(r"\d+\.\d+", val)
        if match:
          rating_val = float(match.group(0))
          return rating_val

    return None

  except Exception as e:
    return None

# 1. Prepare the top 100 products
top_100_products = (
    df.sort_values(by="Review_Count", ascending=False)
    .head(100)[["Product Name", "Brand", "Review_Count", "Rating"]]
    .reset_index(drop=True)
)

top_100_products["Skinsort_Rating"] = None

# 2. Scrape with a persistent session and polite rate limiting
session = requests.Session()

for idx, row in top_100_products.iterrows():
  brand = row["Brand"]
  product = row["Product Name"]

  rating = scrape_skinsort_rating(brand, product, session)
  top_100_products.at[idx, "Skinsort_Rating"] = rating

  # Pause 1-2 seconds between requests to avoid IP bans / 429 Too Many Requests
  time.sleep(1.5)

print("Scraping completed!")
print(top_100_products.head(10))

Scraping completed!
                                    Product Name             Brand  \
0  Relief Sun: Rice + Niacinamide SPF 50+ PA++++  BEAUTY OF JOSEON   
1          Advanced Snail 96 Mucin Power Essence             COSRX   
2                Glow Serum Propolis+Niacinamide  BEAUTY OF JOSEON   
3                                  Dynasty Cream  BEAUTY OF JOSEON   
4                               345 Relief Cream         DR.ALTHEA   
5                      Master Patch Original Fit             COSRX   
6           Heartleaf Pore Control Cleansing Oil              ANUA   
7           Revive Eye Serum : Ginseng + Retinal  BEAUTY OF JOSEON   
8                   Heartleaf 77% Soothing Toner              ANUA   
9           Aloe Soothing Sun Cream SPF50+ PA+++             COSRX   

   Review_Count  Rating Skinsort_Rating  
0        7298.0    4.75            None  
1        2736.0    4.65             3.4  
2        2663.0    4.60            None  
3        2645.0    4.62             2.8  

In [8]:
# 1. Advanced Slug Generator
def slugify(text):
  text = str(text).lower().strip()
  # Normalize SPF ratings and PA indicators (e.g., spf 50+, pa++++)
  text = re.sub(r"pa\++", "", text)
  text = re.sub(r"\+", "", text)
  # Remove special characters, slashes, punctuation
  text = re.sub(r"[^\w\s-]", "", text)
  # Convert multiple spaces/hyphens to a single dash
  text = re.sub(r"[\s_-]+", "-", text).strip("-")
  return text


# 2. Rating Extractor from HTML
def extract_rating(html_text):
  soup = BeautifulSoup(html_text, "html.parser")
  # Target the anchor linking to /reviews
  review_anchor = soup.find("a", href=lambda h: h and "/reviews" in h)

  if review_anchor:
    rating_span = review_anchor.find(
        "span", class_=lambda c: c and "font-semibold" in c
    )
    if rating_span and rating_span.get_text(strip=True):
      try:
        return float(rating_span.get_text(strip=True))
      except ValueError:
        pass

    # Regex fallback within anchor text
    match = re.search(r"\b\d+\.\d+\b", review_anchor.get_text())
    if match:
      return float(match.group(0))

  return None


# 3. Main Scraper with Fallback Search
def scrape_skinsort_rating(brand, product_name, session, headers):
  brand_slug = slugify(brand)
  product_slug = slugify(product_name)

  # Try direct URL pattern first
  direct_url = f"https://skinsort.com/products/{brand_slug}/{product_slug}"

  try:
    resp = session.get(direct_url, headers=headers, timeout=10)
    if resp.status_code == 200:
      rating = extract_rating(resp.text)
      if rating is not None:
        return rating

    # If direct URL returns 404 or no rating, use search fallback
    if resp.status_code == 404 or rating is None:
      search_query = f"{brand} {product_name}".replace(" ", "+")
      search_url = f"https://skinsort.com/products/search?query={search_query}"
      search_resp = session.get(search_url, headers=headers, timeout=10)

      if search_resp.status_code == 200:
        search_soup = BeautifulSoup(search_resp.text, "html.parser")
        first_result = search_soup.find(
            "a", href=lambda h: h and h.startswith("/products/")
        )
        if first_result:
          canonical_url = "https://skinsort.com" + first_result["href"]
          prod_resp = session.get(canonical_url, headers=headers, timeout=10)
          if prod_resp.status_code == 200:
            return extract_rating(prod_resp.text)

    return None

  except Exception:
    return None


# --- Execution Setup ---
mean_reviews = df["Review_Count"].mean()
filtered_df = df[df["Review_Count"] > mean_reviews].copy()

top_100_rating = (
    filtered_df.sort_values(
        by=["Rating", "Review_Count"], ascending=[False, False]
    )
    .head(100)[["Product Name", "Brand", "Review_Count", "Rating"]]
    .reset_index(drop=True)
)

top_100_rating["Skinsort_Rating"] = None

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()

# 4. Scraping Loop with KeyboardInterrupt Safety
try:
  print("Starting scrape for Top 100 products...")
  for idx, row in top_100_rating.iterrows():
    brand = row["Brand"]
    product = row["Product Name"]

    rating = scrape_skinsort_rating(brand, product, session, headers)
    top_100_rating.at[idx, "Skinsort_Rating"] = rating

    print(
        f"[{idx+1}/100] {brand} - {product[:35]}... -> SkinSort Rating:"
        f" {rating}"
    )
    time.sleep(1.2)

except KeyboardInterrupt:
  print("\n🛑 Execution paused or interrupted by user!")

finally:
  top_100_rating.to_csv("top_100_skinsort_scraped.csv", index=False)
  valid_count = top_100_rating["Skinsort_Rating"].notna().sum()
  print(
      f"\n✅ Checkpoint saved to 'top_100_skinsort_scraped.csv' ({valid_count}/100"
      " ratings captured)."
  )

Starting scrape for Top 100 products...
[1/100] HOLIKA HOLIKA - Pure Essence Mask Sheet... -> SkinSort Rating: 4.8
[2/100] BARULAB - The Clean Vegan Hyaluron Mask... -> SkinSort Rating: None
[3/100] BARULAB - The Clean Vegan Vitamin C Mask... -> SkinSort Rating: None
[4/100] HARUHARU WONDER - Black Rice Probiotics Barrier Essen... -> SkinSort Rating: None
[5/100] BARULAB - The Clean Vegan AHA/BHA Mask... -> SkinSort Rating: None
[6/100] BEAUTY OF JOSEON - Centella Asiatica Calming Mask... -> SkinSort Rating: 4.2
[7/100] AESTURA - Atobarrier 365 Cream... -> SkinSort Rating: 3.6
[8/100] MIXSOON - Bean Cleansing Oil... -> SkinSort Rating: 3.0
[9/100] NACIFIC - Rice Pepta Collagen Gel Mask... -> SkinSort Rating: 4.0
[10/100] I'M FROM - Mugwort Mask... -> SkinSort Rating: None
[11/100] DR. CEURACLE - Vegan Kombucha Tea Essence... -> SkinSort Rating: 4.2
[12/100] PYUNKANG YUL - Deep Clear Cleansing Balm... -> SkinSort Rating: 4.4
[13/100] BANILA CO - Clean It Zero Calming Cleansing Bal... ->

In [9]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# 1. Canonical Brand Slug Mapping
BRAND_MAP = {
    "HARUHARU WONDER": "haruharu",
    "PURITO SEOUL": "purito",
    "MARY&MAY": "mary-and-may",
    "I'M FROM": "im-from",
    "A’PIEU": "apieu",
    "A'PIEU": "apieu",
    "SKIN1004": "skin-1004",
    "SOME BY MI": "some-by-mi",
    "DR. CEURACLE": "dr-ceuracle",
    "BEAUTY OF JOSEON": "beauty-of-joseon",
    "ROUND LAB": "round-lab",
}


def clean_slug(text):
  text = str(text).lower().strip()
  # Remove hashtags, parenthesis, slashes, plus signs
  text = re.sub(r"[#\(\)\/+\"']", "", text)
  # Strip volumes and units (e.g. 50ml, 100g)
  text = re.sub(r"\b\d+\s*(ml|g|oz|sheets|ea)\b", "", text, flags=re.I)
  # Collapse spaces/hyphens
  text = re.sub(r"[\s_-]+", "-", text).strip("-")
  return text


def get_candidate_urls(brand, product_name):
  # Normalize brand slug
  brand_key = brand.strip().upper()
  brand_slug = BRAND_MAP.get(brand_key, clean_slug(brand))

  urls = []

  # Primary slug
  base_prod_slug = clean_slug(product_name)
  urls.append(f"https://skinsort.com/products/{brand_slug}/{base_prod_slug}")

  # Variation 1: If product name contains hashtags (e.g., #Green Tea Milk)
  if "#" in product_name:
    simplified = product_name.replace("#", "").strip()
    urls.append(
        f"https://skinsort.com/products/{brand_slug}/{clean_slug(simplified)}"
    )

  # Variation 2: Strip sub-descriptors or common suffixes like 'pack', 'mask'
  if "pack" in base_prod_slug:
    urls.append(
        f"https://skinsort.com/products/{brand_slug}/{base_prod_slug.replace('-pack', '')}"
    )

  # Remove duplicates preserving order
  unique_urls = list(dict.fromkeys(urls))
  return unique_urls


def fetch_rating_from_url(url, session):
  try:
    resp = session.get(url, headers=headers, timeout=8)
    if resp.status_code == 200:
      soup = BeautifulSoup(resp.text, "html.parser")
      review_anchor = soup.find("a", href=lambda h: h and "/reviews" in h)
      if review_anchor:
        rating_span = review_anchor.find(
            "span", class_=lambda c: c and "font-semibold" in c
        )
        if rating_span and rating_span.get_text(strip=True):
          try:
            return float(rating_span.get_text(strip=True))
          except ValueError:
            pass

        match = re.search(r"\b\d+\.\d+\b", review_anchor.get_text())
        if match:
          return float(match.group(0))
    return None
  except Exception:
    return None


# --- Run on the 44 missing rows ---
missing_mask = top_100_rating["Skinsort_Rating"].isna()
missing_indices = top_100_rating[missing_mask].index.tolist()

print(f"Testing URL variations on {len(missing_indices)} missing products...")

session = requests.Session()
recovered = 0

for i, idx in enumerate(missing_indices, start=1):
  row = top_100_rating.loc[idx]
  brand = row["Brand"]
  name = row["Product Name"]

  candidate_urls = get_candidate_urls(brand, name)
  found_rating = None

  for candidate in candidate_urls:
    found_rating = fetch_rating_from_url(candidate, session)
    if found_rating is not None:
      top_100_rating.at[idx, "Skinsort_Rating"] = found_rating
      recovered += 1
      print(f"[{i}/{len(missing_indices)}] MATCHED: {brand} - {name[:30]}")
      print(f"   -> URL: {candidate} (Rating: {found_rating})")
      break
    time.sleep(0.5)

  if found_rating is None:
    print(f"[{i}/{len(missing_indices)}] Still 404: {brand} - {name[:30]}")

  time.sleep(1.0)

print(
    f"\nSecond pass finished! Recovered {recovered} ratings. Total missing now:"
    f" {top_100_rating['Skinsort_Rating'].isna().sum()}"
)

Testing URL variations on 44 missing products...
[1/44] Still 404: BARULAB - The Clean Vegan Hyaluron Mask
[2/44] Still 404: BARULAB - The Clean Vegan Vitamin C Mask
[3/44] MATCHED: HARUHARU WONDER - Black Rice Probiotics Barrier 
   -> URL: https://skinsort.com/products/haruharu/black-rice-probiotics-barrier-essence (Rating: 3.7)
[4/44] Still 404: BARULAB - The Clean Vegan AHA/BHA Mask
[5/44] Still 404: I'M FROM - Mugwort Mask
[6/44] Still 404: MARY&MAY - Hyaluronics Serum
[7/44] MATCHED: PURITO SEOUL - Luminous Ceramide Sleeping Pac
   -> URL: https://skinsort.com/products/purito/luminous-ceramide-sleeping-pack (Rating: 5.0)
[8/44] Still 404: TORRIDEN - DIVE-IN Low Molecular Hyaluron
[9/44] Still 404: I'M FROM - Honey Mask
[10/44] Still 404: A’PIEU - Milk One Pack #Green Tea Milk
[11/44] MATCHED: ISNTREE - Hyaluronic Acid Watery Sun Gel
   -> URL: https://skinsort.com/products/isntree/hyaluronic-acid-watery-sun-gel-spf50-pa (Rating: 3.8)
[12/44] MATCHED: ROUND LAB - Birch Juice Moist

In [10]:
# 1. Build a combined lookup dictionary from all verified ratings in either table
all_known_ratings = (
    pd.concat([
        top_100_products.dropna(subset=["Skinsort_Rating"])[
            ["Brand", "Product Name", "Skinsort_Rating"]
        ],
        top_100_rating.dropna(subset=["Skinsort_Rating"])[
            ["Brand", "Product Name", "Skinsort_Rating"]
        ],
    ])
    .drop_duplicates(subset=["Brand", "Product Name"])
    .set_index(["Brand", "Product Name"])["Skinsort_Rating"]
    .to_dict())

# 2. Backfill missing ratings in top_100_products without changing row order or shape
top_100_products["Skinsort_Rating"] = top_100_products["Skinsort_Rating"].fillna(
    top_100_products.apply(
        lambda row: all_known_ratings.get((row["Brand"], row["Product Name"])),
        axis=1,))

# 3. Backfill missing ratings in top_100_rating without changing row order or shape
top_100_rating["Skinsort_Rating"] = top_100_rating["Skinsort_Rating"].fillna(
    top_100_rating.apply(
        lambda row: all_known_ratings.get((row["Brand"], row["Product Name"])),
        axis=1,))

# 4. Check the updated counts
print(f"top_100_products ratings: {top_100_products['Skinsort_Rating'].notna().sum()} / 100")
print(f"top_100_rating ratings:   {top_100_rating['Skinsort_Rating'].notna().sum()} / 100")

top_100_products ratings: 66 / 100
top_100_rating ratings:   72 / 100


In [11]:
# Save both standalone datasets
top_100_products.to_csv("top_100_products_skinsort.csv", index=False)
top_100_rating.to_csv("top_100_rating_skinsort.csv", index=False)

print(
    f"Saved top_100_products ({top_100_products['Skinsort_Rating'].notna().sum()}/100 ratings)."
)
print(
    f"Saved top_100_rating ({top_100_rating['Skinsort_Rating'].notna().sum()}/100 ratings)."
)

Saved top_100_products (66/100 ratings).
Saved top_100_rating (72/100 ratings).
